# Task 1: Create a Prescription Parser using CRF
This task tests your ability to build a Doctor Prescription Parser with the help of CRF model

Your job is to build a Prescription Parser that takes a prescription (sentence) as an input and find / label the words in that sentence with one of the already pre-defined labels

### Problem: SEQUENCE PREDICTION - Label words in a sentence
#### Input : Doctor Prescription in the form of a sentence split into tokens
- Ex: Take 2 tablets once a day for 10 days

#### Output : FHIR Labels
- ('Take', 'Method')
- ('2', 'Qty') 
- ('tablets', 'Form')
- ('once', 'Frequency')
- ('a', 'Period') 
- ('day', 'PeriodUnit')
- ('for', 'FOR')
- ('10', 'Duration')
- ('days', 'DurationUnit') 

### Major Steps
- Install necessary library
- Import the libraries
- Create training data with labels
    - Split the sentence into tokens
    - Compute POS tags
    - Create triples
- Extract features
- Split the data into training and testing set
- Create CRF model
- Save the CRF model
- Load the CRF model
- Predict on test data
- Accuracy

#### Install necessary library

In [3]:
# Install python-crfsuite — the CRF library used for sequence labeling
# Run this cell once, then restart the kernel if needed
# !pip install python-crfsuite nltk scikit-learn

#### Import the necessary libraries

In [4]:
import re
import os
import pycrfsuite
from itertools import chain
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print('All libraries imported successfully!')

All libraries imported successfully!


### Input data (GIVEN)
#### Creating the inputs to the ML model in the following form:
- sigs --> ['take 3 tabs for 10 days']       INPUT SIG
- input_sigs --> [['take', '3', 'tabs', 'for', '10', 'days']]      TOKENS
- output_labels --> [['Method','Qty', 'Form', 'FOR', 'Duration', 'DurationUnit']]       LABELS

In [5]:
sigs = ["for 5 to 6 days", "inject 2 units", "x 2 weeks", "x 3 days", "every day", "every 2 weeks", "every 3 days", "every 1 to 2 months", "every 2 to 6 weeks", "every 4 to 6 days", "take two to four tabs", "take 2 to 4 tabs", "take 3 tabs orally bid for 10 days at bedtime", "swallow three capsules tid orally", "take 2 capsules po every 6 hours", "take 2 tabs po for 10 days", "take 100 caps by mouth tid for 10 weeks", "take 2 tabs after an hour", "2 tabs every 4-6 hours", "every 4 to 6 hours", "q46h", "q4-6h", "2 hours before breakfast", "before 30 mins at bedtime", "30 mins before bed", "and 100 tabs twice a month", "100 tabs twice a month", "100 tabs once a month", "100 tabs thrice a month", "3 tabs daily for 3 days then 1 tab per day at bed", "30 tabs 10 days tid", "take 30 tabs for 10 days three times a day", "qid q6h", "bid", "qid", "30 tabs before dinner and bedtime", "30 tabs before dinner & bedtime", "take 3 tabs at bedtime", "30 tabs thrice daily for 10 days ", "30 tabs for 10 days three times a day", "Take 2 tablets a day", "qid for 10 days", "every day", "take 2 caps at bedtime", "apply 3 drops before bedtime", "take three capsules daily", "swallow 3 pills once a day", "swallow three pills thrice a day", "apply daily", "apply three drops before bedtime", "every 6 hours", "before food", "after food", "for 20 days", "for twenty days", "with meals"]
input_sigs = [['for', '5', 'to', '6', 'days'], ['inject', '2', 'units'], ['x', '2', 'weeks'], ['x', '3', 'days'], ['every', 'day'], ['every', '2', 'weeks'], ['every', '3', 'days'], ['every', '1', 'to', '2', 'months'], ['every', '2', 'to', '6', 'weeks'], ['every', '4', 'to', '6', 'days'], ['take', 'two', 'to', 'four', 'tabs'], ['take', '2', 'to', '4', 'tabs'], ['take', '3', 'tabs', 'orally', 'bid', 'for', '10', 'days', 'at', 'bedtime'], ['swallow', 'three', 'capsules', 'tid', 'orally'], ['take', '2', 'capsules', 'po', 'every', '6', 'hours'], ['take', '2', 'tabs', 'po', 'for', '10', 'days'], ['take', '100', 'caps', 'by', 'mouth', 'tid', 'for', '10', 'weeks'], ['take', '2', 'tabs', 'after', 'an', 'hour'], ['2', 'tabs', 'every', '4-6', 'hours'], ['every', '4', 'to', '6', 'hours'], ['q46h'], ['q4-6h'], ['2', 'hours', 'before', 'breakfast'], ['before', '30', 'mins', 'at', 'bedtime'], ['30', 'mins', 'before', 'bed'], ['and', '100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'once', 'a', 'month'], ['100', 'tabs', 'thrice', 'a', 'month'], ['3', 'tabs', 'daily', 'for', '3', 'days', 'then', '1', 'tab', 'per', 'day', 'at', 'bed'], ['30', 'tabs', '10', 'days', 'tid'], ['take', '30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], ['qid', 'q6h'], ['bid'], ['qid'], ['30', 'tabs', 'before', 'dinner', 'and', 'bedtime'], ['30', 'tabs', 'before', 'dinner', '&', 'bedtime'], ['take', '3', 'tabs', 'at', 'bedtime'], ['30', 'tabs', 'thrice', 'daily', 'for', '10', 'days'], ['30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], ['take', '2', 'tablets', 'a', 'day'], ['qid', 'for', '10', 'days'], ['every', 'day'], ['take', '2', 'caps', 'at', 'bedtime'], ['apply', '3', 'drops', 'before', 'bedtime'], ['take', 'three', 'capsules', 'daily'], ['swallow', '3', 'pills', 'once', 'a', 'day'], ['swallow', 'three', 'pills', 'thrice', 'a', 'day'], ['apply', 'daily'], ['apply', 'three', 'drops', 'before', 'bedtime'], ['every', '6', 'hours'], ['before', 'food'], ['after', 'food'], ['for', '20', 'days'], ['for', 'twenty', 'days'], ['with', 'meals']]
output_labels = [['FOR', 'Duration', 'TO', 'DurationMax', 'DurationUnit'], ['Method', 'Qty', 'Form'], ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['EVERY', 'Period'], ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'Form', 'PO', 'BID', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'TID', 'PO'], ['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'PO', 'FOR', 'Duration', 'DurationUnit'], ['Method', 'Qty', 'Form', 'BY', 'PO', 'TID', 'FOR', 'Duration', 'DurationUnit'], ['Method', 'Qty', 'Form', 'AFTER', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Q46H'], ['Q4-6H'], ['Qty', 'PeriodUnit', 'BEFORE', 'WHEN'], ['BEFORE', 'Qty', 'M', 'AT', 'WHEN'], ['Qty', 'M', 'BEFORE', 'WHEN'], ['AND', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'FOR', 'Duration', 'DurationUnit', 'THEN', 'Qty', 'Form', 'Frequency', 'PeriodUnit', 'AT', 'WHEN'], ['Qty', 'Form', 'Duration', 'DurationUnit', 'TID'], ['Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Qty', 'TIMES', 'Period', 'PeriodUnit'], ['QID', 'Q6H'], ['BID'], ['QID'],['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Qty', 'Form', 'Frequency', 'DAILY', 'FOR', 'Duration', 'DurationUnit'], ['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Frequency', 'TIMES', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'Period', 'PeriodUnit'], ['QID', 'FOR', 'Duration', 'DurationUnit'], ['EVERY', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['Method', 'Qty', 'Form', 'DAILY'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'DAILY'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['EVERY', 'Period', 'PeriodUnit'], ['BEFORE', 'FOOD'], ['AFTER', 'FOOD'], ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['WITH', 'FOOD']]

In [6]:
len(sigs), len(input_sigs) , len(output_labels)

(56, 56, 56)

### Helper: Rule-based POS Tagger
We use a lightweight rule-based POS tagger designed specifically for medical prescription tokens.
This avoids a dependency on NLTK data downloads while still providing meaningful part-of-speech features.

In [7]:
def simple_pos_tag(tokens):
    """
    Rule-based POS tagger optimised for prescription/SIG text.
    Returns a list of (token, POS_tag) tuples, mimicking NLTK's pos_tag output.
    Tags used: CD (cardinal), VB (verb), NNS/NN (nouns), DT (determiner),
               IN (preposition), TO, CC (conjunction), RB (adverb), JJ (adjective)
    """
    NUMBER_WORDS = {
        'one','two','three','four','five','six','seven','eight','nine','ten',
        'twenty','thirty','forty','fifty','sixty','hundred'
    }
    FORM_PLURALS  = {'tabs','tablets','capsules','caps','pills','drops','units','mins'}
    FORM_SINGULAR = {'tab','capsule','pill','drop','unit','hour','day','week','month',
                     'meal','bed','breakfast','dinner','bedtime','food','morning',
                     'evening','mouth'}
    TIME_PLURALS  = {'hours','days','weeks','months','times','meals'}
    DETERMINERS   = {'every','a','an','the','each'}
    PREPS         = {'for','at','in','with','by','after','before','of'}
    ADVERBS       = {'orally','daily','once','twice','thrice','then','per','please','x'}
    VERBS         = {'take','swallow','inject','apply'}
    ABBREV_NN     = {'bid','tid','qid','po','pu','q6h','q46h','q4-6h','m'}

    tagged = []
    for token in tokens:
        t = token.lower()

        if re.fullmatch(r'\d+', token) or t in NUMBER_WORDS:
            tag = 'CD'
        elif re.fullmatch(r'\d+-\d+', token):   # e.g. 4-6
            tag = 'JJ'
        elif re.fullmatch(r'q\d+h', t) or re.fullmatch(r'q\d+-\d+h', t):
            tag = 'NN'
        elif t in VERBS:          tag = 'VB'
        elif t in FORM_PLURALS:   tag = 'NNS'
        elif t in FORM_SINGULAR:  tag = 'NN'
        elif t in TIME_PLURALS:   tag = 'NNS'
        elif t in DETERMINERS:    tag = 'DT'
        elif t in PREPS:          tag = 'IN'
        elif t == 'to':           tag = 'TO'
        elif t == 'and':          tag = 'CC'
        elif t == '&':            tag = 'CC'
        elif t in ADVERBS:        tag = 'RB'
        elif t in ABBREV_NN:      tag = 'NN'
        elif t.endswith('ly'):    tag = 'RB'
        else:                     tag = 'NN'

        tagged.append((token, tag))
    return tagged


# Quick sanity check
print(simple_pos_tag(['take', '2', 'tabs', 'every', '6', 'hours']))

[('take', 'VB'), ('2', 'CD'), ('tabs', 'NNS'), ('every', 'DT'), ('6', 'CD'), ('hours', 'NNS')]


### Creating a Tuples Maker method
Create the tuples as given below by writing a function **tuples_maker(input_sigs, output_labels)** and returns **output** as given below

Input(s): 
- input_sigs
- output_lables

Output:

[[('for', 'FOR'),
  ('5', 'Duration'),
  ('to', 'TO'),
  ('6', 'DurationMax'),
  ('days', 'DurationUnit')], [second sentence], ...]

In [8]:
def tuples_maker(inp, out):
    """
    Zips each list of tokens with its corresponding list of labels
    to produce (token, label) pairs for every sentence.

    Parameters
    ----------
    inp : list of list of str   – tokenised input sigs
    out : list of list of str   – FHIR label sequences

    Returns
    -------
    sample_data : list of list of (str, str) tuples
    """
    sample_data = [
        list(zip(tokens, labels))
        for tokens, labels in zip(inp, out)
    ]
    return sample_data

In [9]:
# Build (token, label) tuples and display the first few sentences
whole_data = tuples_maker(input_sigs, output_labels)
whole_data[:5]

[[('for', 'FOR'),
  ('5', 'Duration'),
  ('to', 'TO'),
  ('6', 'DurationMax'),
  ('days', 'DurationUnit')],
 [('inject', 'Method'), ('2', 'Qty'), ('units', 'Form')],
 [('x', 'FOR'), ('2', 'Duration'), ('weeks', 'DurationUnit')],
 [('x', 'FOR'), ('3', 'Duration'), ('days', 'DurationUnit')],
 [('every', 'EVERY'), ('day', 'Period')]]

### Creating the triples_maker( ) for feature extraction
- input: tuples_maker_output
- output: 
[[('for', 'IN', 'FOR'),
  ('5', 'CD', 'Duration'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'DurationMax'),
  ('days', 'NNS', 'DurationUnit')], [second sentence], ... ]

In [10]:
def triples_maker(whole_data):
    """
    Enriches each (token, label) pair with a POS tag to produce
    (token, POS, label) triples.  The POS tag is computed using
    the rule-based simple_pos_tag() function defined above.

    Parameters
    ----------
    whole_data : output of tuples_maker() – list of [(token, label)]

    Returns
    -------
    sample_data : list of list of (token, POS, label) triples
    """
    sample_data = []
    for sentence in whole_data:
        tokens = [tok for (tok, _lbl) in sentence]
        labels = [lbl for (_tok, lbl) in sentence]
        pos_tagged = simple_pos_tag(tokens)                        # [(tok, pos), ...]
        triples = [
            (tok, pos, lbl)
            for (tok, pos), lbl in zip(pos_tagged, labels)
        ]
        sample_data.append(triples)
    return sample_data

In [11]:
sample_data = triples_maker(whole_data)
sample_data

[[('for', 'IN', 'FOR'),
  ('5', 'CD', 'Duration'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'DurationMax'),
  ('days', 'NNS', 'DurationUnit')],
 [('inject', 'VB', 'Method'), ('2', 'CD', 'Qty'), ('units', 'NNS', 'Form')],
 [('x', 'RB', 'FOR'),
  ('2', 'CD', 'Duration'),
  ('weeks', 'NNS', 'DurationUnit')],
 [('x', 'RB', 'FOR'),
  ('3', 'CD', 'Duration'),
  ('days', 'NNS', 'DurationUnit')],
 [('every', 'DT', 'EVERY'), ('day', 'NN', 'Period')],
 [('every', 'DT', 'EVERY'),
  ('2', 'CD', 'Period'),
  ('weeks', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('3', 'CD', 'Period'),
  ('days', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('1', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('2', 'CD', 'PeriodMax'),
  ('months', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('2', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'PeriodMax'),
  ('weeks', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('4', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'PeriodMax'),
  ('

### Creating the features extractor method (GIVEN as a BASELINE)
#### The features used are:
- SOS, EOS, lowercase, uppercase, title, digit, postag, previous_tag, next_tag
#### Feel free to include more features

In [12]:
def token_to_features(doc, i):
    word = doc[i][0]
    postag = doc[i][1]

    # Common features for all words
    features = [
        'bias',
        'word.lower=' + word.lower(),
        'word[-3:]=' + word[-3:],
        'word[-2:]=' + word[-2:],
        'word.isupper=%s' % word.isupper(),
        'word.istitle=%s' % word.istitle(),
        'word.isdigit=%s' % word.isdigit(),
        'postag=' + postag
    ]

    # Features for words that are not
    # at the beginning of a document
    if i > 0:
        word1 = doc[i-1][0]
        postag1 = doc[i-1][1]
        features.extend([
            '-1:word.lower=' + word1.lower(),
            '-1:word.istitle=%s' % word1.istitle(),
            '-1:word.isupper=%s' % word1.isupper(),
            '-1:word.isdigit=%s' % word1.isdigit(),
            '-1:postag=' + postag1
        ])
    else:
        # Indicate that it is the 'beginning of a document'
        features.append('BOS')

    # Features for words that are not
    # at the end of a document
    if i < len(doc)-1:
        word1 = doc[i+1][0]
        postag1 = doc[i+1][1]
        features.extend([
            '+1:word.lower=' + word1.lower(),
            '+1:word.istitle=%s' % word1.istitle(),
            '+1:word.isupper=%s' % word1.isupper(),
            '+1:word.isdigit=%s' % word1.isdigit(),
            '+1:postag=' + postag1
        ])
    else:
        # Indicate that it is the 'end of a document'
        features.append('EOS')

    return features

### Running the feature extractor on the training data 
- Feature extraction
- Train-test-split

In [13]:
from sklearn.model_selection import train_test_split

# A function for extracting features in documents
def get_features(doc):
    return [token_to_features(doc, i) for i in range(len(doc))]

# A function fo generating the list of labels for each document
def get_labels(doc):
    return [label for (token, postag, label) in doc]

#[print(doc) for doc in sample_data]
X = [get_features(doc) for doc in sample_data]

y = [get_labels(doc) for doc in sample_data]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

Training samples : 44
Test samples     : 12


### Training the CRF model with the features extracted using the feature extractor method

In [14]:


# Submit training data to the trainer
trainer = pycrfsuite.Trainer(verbose=True)

for xseq, yseq in zip(X_train, y_train):
    trainer.append(xseq, yseq)

# Set the parameters of the model
trainer.set_params({
    'c1': 0.1,                          # L1 regularisation coefficient
    'c2': 0.01,                         # L2 regularisation coefficient
    'max_iterations': 1000,             # stop after 1000 iterations
    'feature.possible_transitions': True  # include all possible label transitions
})

# Providing a file name as a parameter to the train function, such that
# the model will be saved to the file when training is finished
trainer.train('prescription_crf.model')

print('\nModel saved to: prescription_crf.model')

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 1505
Seconds required: 0.005

L-BFGS optimization
c1: 0.100000
c2: 0.010000
num_memories: 6
max_iterations: 1000
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 536.757419
Feature norm: 1.000000
Error norm: 113.162023
Active features: 1499
Line search trials: 1
Line search step: 0.006574
Seconds required for this iteration: 0.001

***** Iteration #2 *****
Loss: 342.247389
Feature norm: 4.984196
Error norm: 141.425629
Active features: 1442
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.004

***** Iteration #3 *****
Loss: 238.985389
Feature norm: 6.341337
Error norm: 69.056413
Active features: 1437
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration:

### Predicting the test data with the built model

In [15]:
# Load the saved model
tagger = pycrfsuite.Tagger()
tagger.open('prescription_crf.model')
print('Model loaded successfully.')

Model loaded successfully.


In [16]:
# Predict on the test set
y_pred = [tagger.tag(xseq) for xseq in X_test]

print('Predictions on test set:')
for true_seq, pred_seq in zip(y_test, y_pred):
    print(f'  True : {true_seq}')
    print(f'  Pred : {pred_seq}')
    print()

Predictions on test set:
  True : ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit']
  Pred : ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit']

  True : ['FOR', 'Duration', 'DurationUnit']
  Pred : ['FOR', 'Duration', 'DurationUnit']

  True : ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']
  Pred : ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']

  True : ['Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Qty', 'TIMES', 'Period', 'PeriodUnit']
  Pred : ['Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Frequency', 'TIMES', 'Period', 'PeriodUnit']

  True : ['FOR', 'Duration', 'TO', 'DurationMax', 'DurationUnit']
  Pred : ['FOR', 'Duration', 'TO', 'PeriodMax', 'PeriodUnit']

  True : ['BID']
  Pred : ['BID']

  True : ['Qty', 'Form', 'Frequency', 'FOR', 'Duration', 'DurationUnit', 'THEN', 'Qty', 'Form', 'Frequency', 'PeriodUnit', 'AT', 'WHEN']
  Pred : ['Qty', 'Form', 'DAILY', 'FOR', 'Duration', 'DurationUnit', 'FOR', 'Duration'

In [17]:
# ── Token-level Accuracy ───────────────────────────────────────────────────────
all_true_labels = list(chain.from_iterable(y_test))
all_pred_labels = list(chain.from_iterable(y_pred))

correct = sum(t == p for t, p in zip(all_true_labels, all_pred_labels))
total   = len(all_true_labels)
token_accuracy = correct / total

print(f'Token-level Accuracy : {token_accuracy * 100:.2f}%  ({correct}/{total} tokens correct)')

Token-level Accuracy : 75.86%  (44/58 tokens correct)


In [18]:
# ── Sentence-level Accuracy ────────────────────────────────────────────────────
sent_correct = sum(1 for t, p in zip(y_test, y_pred) if t == p)
sent_total   = len(y_test)
sent_accuracy = sent_correct / sent_total

print(f'Sentence-level Accuracy : {sent_accuracy * 100:.2f}%  ({sent_correct}/{sent_total} sentences fully correct)')

Sentence-level Accuracy : 41.67%  (5/12 sentences fully correct)


In [19]:
# ── Per-label Classification Report ───────────────────────────────────────────
# Collect all unique labels from both true and predicted
all_labels_set = sorted(set(all_true_labels + all_pred_labels))

print('Per-label Classification Report:\n')
print(classification_report(
    all_true_labels,
    all_pred_labels,
    labels=all_labels_set,
    zero_division=0
))

Per-label Classification Report:

              precision    recall  f1-score   support

       AFTER       0.00      0.00      0.00         1
          AT       1.00      1.00      1.00         1
         BID       1.00      1.00      1.00         1
       DAILY       0.00      0.00      0.00         0
    Duration       0.67      1.00      0.80         4
 DurationMax       0.00      0.00      0.00         1
DurationUnit       0.60      0.75      0.67         4
       EVERY       1.00      1.00      1.00         2
        FOOD       0.00      0.00      0.00         1
         FOR       0.80      1.00      0.89         4
        Form       0.80      0.80      0.80         5
   Frequency       0.33      0.33      0.33         3
      Method       1.00      1.00      1.00         3
      Period       1.00      1.00      1.00         5
   PeriodMax       0.67      1.00      0.80         2
  PeriodUnit       0.83      0.83      0.83         6
       Q4-6H       0.00      0.00      0.00    

In [20]:
# ── Top positive / negative features learned by the model ─────────────────────
info = tagger.info()

def print_top_transitions(trans_features, header):
    print(f'\n{header}')
    for (label_from, label_to), weight in trans_features:
        print(f'  {weight:+.4f}  {label_from} --> {label_to}')

print_top_transitions(
    sorted(info.transitions.items(), key=lambda x: x[1], reverse=True)[:10],
    'Top-10 LIKELY transitions:'
)
print_top_transitions(
    sorted(info.transitions.items(), key=lambda x: x[1])[:10],
    'Top-10 UNLIKELY transitions:'
)


Top-10 LIKELY transitions:
  +5.3945  Duration --> DurationUnit
  +4.6219  FOR --> Duration
  +3.9826  Qty --> Form
  +3.2702  Period --> PeriodUnit
  +2.4453  PeriodMax --> PeriodUnit
  +2.3858  Method --> Qty
  +2.3844  BY --> PO
  +2.3414  EVERY --> Period
  +2.0567  Frequency --> TIMES
  +1.9369  Qty --> M

Top-10 UNLIKELY transitions:
  -0.2124  Form --> Qty
  +0.0386  AND --> Qty
  +0.1097  DurationUnit --> TID
  +0.1949  Method --> DAILY
  +0.2816  EVERY --> PeriodUnit
  +0.3618  BEFORE --> Qty
  +0.3638  Form --> BY
  +0.4250  BID --> FOR
  +0.4360  Form --> Frequency
  +0.4892  PeriodUnit --> BEFORE


In [21]:
# Confirm model file exists on disk
model_path = 'prescription_crf.model'
size_kb = os.path.getsize(model_path) / 1024
print(f'Model file: {model_path}  ({size_kb:.1f} KB)')

Model file: prescription_crf.model  (18.6 KB)


### Putting all the prediction logic inside a predict method

In [22]:
def predict(sig):
    """
    predict(sig)
    Purpose: Labels the given sig into corresponding labels
    @param sig. A Sentence  # A medical prescription sig written by a doctor
    @return     A list      # A list with predicted labels (first level of labeling)
    >>> predict('2 tabs every 4 hours')
    [['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit']]
    >>> predict('2 tabs with food')
    [['Qty', 'Form', 'WITH', 'FOOD']]
    >>> predict('2 tabs qid x 30 days')
    [['Qty', 'Form', 'QID', 'FOR', 'Duration', 'DurationUnit']]
    """
    # 1. Tokenise by splitting on whitespace
    tokens = sig.split()

    # 2. POS-tag the tokens
    pos_tagged = simple_pos_tag(tokens)            # [(token, POS), ...]

    # 3. Build (token, POS, placeholder_label) triples expected by get_features()
    triples = [(tok, pos, 'O') for (tok, pos) in pos_tagged]

    # 4. Extract CRF features
    features = get_features(triples)

    # 5. Run the CRF tagger and wrap result in a list (to match expected output format)
    predictions = [tagger.tag(features)]

    # 6. Print input and prediction for readability
    print(sig)
    print(predictions)

    return predictions

### Sample predictions

In [23]:
predictions = predict("take 2 tabs every 6 hours x 10 days")

take 2 tabs every 6 hours x 10 days
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit', 'FOR', 'Duration', 'DurationUnit']]


In [24]:
predictions = predict("2 capsu for 10 day at bed")

2 capsu for 10 day at bed
[['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN']]


In [25]:
predictions = predict("2 capsu for 10 days at bed")

2 capsu for 10 days at bed
[['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN']]


In [26]:
predictions = predict("5 days 2 tabs at bed")

5 days 2 tabs at bed
[['Duration', 'DurationUnit', 'Qty', 'Form', 'AT', 'WHEN']]


In [27]:
predictions = predict("3 tabs qid x 10 weeks")

3 tabs qid x 10 weeks
[['Qty', 'Form', 'QID', 'FOR', 'Duration', 'DurationUnit']]


In [28]:
predictions = predict("x 30 days")

x 30 days
[['FOR', 'Duration', 'DurationUnit']]


In [29]:
predictions = predict("x 20 months")

x 20 months
[['FOR', 'Duration', 'DurationUnit']]


In [30]:
predictions = predict("take 2 tabs po tid for 10 days")

take 2 tabs po tid for 10 days
[['Method', 'Qty', 'Form', 'PO', 'TID', 'FOR', 'Duration', 'DurationUnit']]


In [31]:
predictions = predict("take 2 capsules po every 6 hours")

take 2 capsules po every 6 hours
[['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit']]


In [32]:
predictions = predict("inject 2 units pu tid")

inject 2 units pu tid
[['Method', 'Qty', 'Form', 'PO', 'TID']]


In [33]:
predictions = predict("swallow 3 caps tid by mouth")

swallow 3 caps tid by mouth
[['Method', 'Qty', 'Form', 'TID', 'BY', 'PO']]


In [34]:
predictions = predict("inject 3 units orally")

inject 3 units orally
[['Method', 'Qty', 'Form', 'PO']]


In [35]:
predictions = predict("orally take 3 tabs tid")

orally take 3 tabs tid
[['PO', 'Method', 'Qty', 'Form', 'TID']]


In [36]:
predictions = predict("by mouth take three caps")

by mouth take three caps
[['BY', 'PO', 'Method', 'Qty', 'Form']]


In [37]:
predictions = predict("take 3 tabs orally three times a day for 10 days at bedtime")

take 3 tabs orally three times a day for 10 days at bedtime
[['Method', 'Qty', 'Form', 'PO', 'Frequency', 'TIMES', 'Period', 'PeriodUnit', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN']]


In [38]:
predictions = predict("take 3 tabs orally bid for 10 days at bedtime")

take 3 tabs orally bid for 10 days at bedtime
[['Method', 'Qty', 'Form', 'PO', 'BID', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN']]


In [39]:
predictions = predict("take 3 tabs bid orally at bed")

take 3 tabs bid orally at bed
[['Method', 'Qty', 'Form', 'BY', 'PO', 'AT', 'WHEN']]


In [40]:
predictions = predict("take 10 capsules by mouth qid")

take 10 capsules by mouth qid
[['Method', 'Qty', 'Form', 'BY', 'PO', 'QID']]


In [41]:
predictions = predict("inject 10 units orally qid x 3 months")

inject 10 units orally qid x 3 months
[['Method', 'Qty', 'Form', 'PO', 'QID', 'FOR', 'Duration', 'DurationUnit']]


In [42]:
prediction = predict("please take 2 tablets per day for a month in the morning and evening each day")

please take 2 tablets per day for a month in the morning and evening each day
[['Qty', 'Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Period', 'PeriodUnit', 'FOR', 'Duration', 'DurationUnit', 'AND', 'WHEN', 'Period', 'PeriodUnit']]


In [43]:
prediction = predict("Amoxcicillin QID 30 tablets")

Amoxcicillin QID 30 tablets
[['Duration', 'DurationUnit', 'Qty', 'Form']]


In [44]:
prediction = predict("take 3 tabs TID for 90 days with food")

take 3 tabs TID for 90 days with food
[['Method', 'Qty', 'Form', 'TID', 'FOR', 'Duration', 'DurationUnit', 'AFTER', 'FOOD']]


In [45]:
prediction = predict("with food take 3 tablets per day for 90 days")

with food take 3 tablets per day for 90 days
[['AFTER', 'FOOD', 'Method', 'Qty', 'Form', 'Duration', 'DurationUnit', 'FOR', 'Duration', 'DurationUnit']]


In [46]:
prediction = predict("with food take 3 tablets per week for 90 weeks")

with food take 3 tablets per week for 90 weeks
[['AFTER', 'FOOD', 'Method', 'Qty', 'Form', 'Duration', 'DurationUnit', 'FOR', 'Duration', 'DurationUnit']]


In [47]:
prediction = predict("take 2-4 tabs")

take 2-4 tabs
[['Method', 'Qty', 'Form']]


In [48]:
prediction = predict("take 2 to 4 tabs")

take 2 to 4 tabs
[['Method', 'Qty', 'TO', 'Qty', 'Form']]


In [49]:
prediction = predict("take two to four tabs")

take two to four tabs
[['Method', 'Qty', 'TO', 'Qty', 'Form']]


In [50]:
prediction = predict("take 2-4 tabs for 8 to 9 days")

take 2-4 tabs for 8 to 9 days
[['Method', 'Qty', 'Form', 'FOR', 'Duration', 'TO', 'PeriodMax', 'PeriodUnit']]


In [51]:
prediction = predict("take 20 tabs every 6 to 8 days")

take 20 tabs every 6 to 8 days
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [52]:
prediction = predict("take 2 tabs every 4 to 6 days")

take 2 tabs every 4 to 6 days
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [53]:
prediction = predict("take 2 tabs every 2 to 10 weeks")

take 2 tabs every 2 to 10 weeks
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [54]:
prediction = predict("take 2 tabs every 4 to 6 days")

take 2 tabs every 4 to 6 days
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [55]:
prediction = predict("take 2 tabs every 2 to 10 months")

take 2 tabs every 2 to 10 months
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [56]:
prediction = predict("every 60 mins")

every 60 mins
[['EVERY', 'Period', 'PeriodUnit']]


In [57]:
prediction = predict("every 10 mins")

every 10 mins
[['EVERY', 'Period', 'PeriodUnit']]


In [58]:
prediction = predict("every two to four months")

every two to four months
[['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [59]:
prediction = predict("take 2 tabs every 3 to 4 days")

take 2 tabs every 3 to 4 days
[['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit']]


In [60]:
prediction = predict("every 3 to 4 days take 20 tabs")

every 3 to 4 days take 20 tabs
[['EVERY', 'Period', 'TO', 'Duration', 'DurationUnit', 'Method', 'Qty', 'Form']]


In [61]:
prediction = predict("once in every 3 days take 3 tabs")

once in every 3 days take 3 tabs
[['Frequency', 'TIMES', 'EVERY', 'Period', 'PeriodUnit', 'Method', 'Qty', 'Form']]


In [62]:
prediction = predict("take 3 tabs once in every 3 days")

take 3 tabs once in every 3 days
[['Method', 'Qty', 'Form', 'Frequency', 'TIMES', 'EVERY', 'Period', 'PeriodUnit']]


In [63]:
prediction = predict("orally take 20 tabs every 4-6 weeks")

orally take 20 tabs every 4-6 weeks
[['PO', 'Method', 'Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit']]


In [64]:
prediction = predict("10 tabs x 2 days")

10 tabs x 2 days
[['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit']]


In [65]:
prediction = predict("3 capsule x 15 days")

3 capsule x 15 days
[['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit']]


In [66]:
prediction = predict("10 tabs")

10 tabs
[['Qty', 'Form']]
